# Importando bibliotecas

In [1]:
import pandas as pd
import plotly.express as px
from ipywidgets import Dropdown, VBox, HBox, Output
from IPython.display import display

# Importando dados

In [2]:
df = pd.read_excel('../data/raw/Relatório de Produção - Completo - Bloco de Notas.xlsx')

c:\Users\evosystem03.ti\Documents\Demanda Carteira\Projeto\predicao-carteira-wheaton\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Visualizando dados

In [3]:
df

,Bloco de Notas - Geral,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Turno Wht,NaN,Maquina,Hora,Funcao,Autor,Nota,Prefixo
2,Manhã,NaN,A1,2026-08-10 06:05:07,Operador AQ,AQA1 Erik Kleiton,"Ferramentas, Equipamentos, Calibres, Rejeitore...",SB -1035-SWN
3,Manhã,NaN,A1,2026-08-10 06:25:17,Esc. Automatica,VINICIUS,"CANAIS MAQ MX4\nGL1:RTE,RAB,RAB\nGL3:RRS,RGA,R...",SB -1035-SWN
4,Manhã,NaN,A1,2026-08-10 08:03:34,Lider AF,Lima,"Ás 08hrs 2 PCTS sem apontar.(2,75%)",SB -1035-SWN
...,...,...,...,...,...,...,...,...
282,Noite,NaN,A5,2026-08-10 04:45:38,Lider de turno,FABIO,10/08/2026\n\nMÁQUINA A5 VEL: 242\nDRT: 10029...,SB -1290-S
283,Noite,NaN,A5,2026-08-10 22:07:13,Operador AQ,AQA5 DANILO BATISTA,QUADRO OK\nCALIBRES OK,SB -1290-S
284,Noite,NaN,A5,2026-08-10 22:08:15,Esc. Automatica,PATRICK,"CANAIS MAQ MX4\nGL1:RTE,RAB,RAB\nGL3:RRS,RRS,R...",SB -1290-S
285,Noite,NaN,A6,2026-08-10 04:43:53,Lider de turno,FABIO,10/08/2026\n\nMÁQUINA A6 VEL: 170\nDRT: 9363\...,SB -1579-N1N


# Tratando dados

In [4]:
# tratamentos df
display(df.info())

# Removendo coluna só com valores nan
df = df.drop(columns=['Unnamed: 1'])

# ajustando cabeçalho da base
df.columns = df.iloc[1]
df = df.iloc[2:].reset_index(drop=True)
df.columns.name = None

# Removendo espaços nos valores de "Prefixo"
df['Prefixo'] = df['Prefixo'].str.replace(' ', '').copy()

# Renomeando coluna "Hora" para "Data", melhor compreensão
df.rename(columns={'Hora': 'Data'}, inplace=True)

display(df)

<class 'pandas.DataFrame'>
RangeIndex: 287 entries, 0 to 286
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Bloco de Notas - Geral  286 non-null    str    
 1   Unnamed: 1              0 non-null      float64
 2   Unnamed: 2              286 non-null    str    
 3   Unnamed: 3              286 non-null    object 
 4   Unnamed: 4              286 non-null    str    
 5   Unnamed: 5              286 non-null    str    
 6   Unnamed: 6              286 non-null    str    
 7   Unnamed: 7              286 non-null    str    
dtypes: float64(1), object(1), str(6)
memory usage: 58.5+ KB


None

,Turno Wht,Maquina,Data,Funcao,Autor,Nota,Prefixo
0,Manhã,A1,2026-08-10 06:05:07,Operador AQ,AQA1 Erik Kleiton,"Ferramentas, Equipamentos, Calibres, Rejeitore...",SB-1035-SWN
1,Manhã,A1,2026-08-10 06:25:17,Esc. Automatica,VINICIUS,"CANAIS MAQ MX4\nGL1:RTE,RAB,RAB\nGL3:RRS,RGA,R...",SB-1035-SWN
2,Manhã,A1,2026-08-10 08:03:34,Lider AF,Lima,"Ás 08hrs 2 PCTS sem apontar.(2,75%)",SB-1035-SWN
3,Manhã,A1,2026-08-10 08:43:23,Lider AF,Lima,"Amostra MCAL4 10 frascos (BOA,RCO,DOB) 5,6%",SB-1035-SWN
4,Manhã,A1,2026-08-10 13:17:00,Operador AQ,Dantas 2565,Das 12:04 as 12:10 seção 3 parada devido a ent...,SB-1035-SWN
...,...,...,...,...,...,...,...
280,Noite,A5,2026-08-10 04:45:38,Lider de turno,FABIO,10/08/2026\n\nMÁQUINA A5 VEL: 242\nDRT: 10029...,SB-1290-S
281,Noite,A5,2026-08-10 22:07:13,Operador AQ,AQA5 DANILO BATISTA,QUADRO OK\nCALIBRES OK,SB-1290-S
282,Noite,A5,2026-08-10 22:08:15,Esc. Automatica,PATRICK,"CANAIS MAQ MX4\nGL1:RTE,RAB,RAB\nGL3:RRS,RRS,R...",SB-1290-S
283,Noite,A6,2026-08-10 04:43:53,Lider de turno,FABIO,10/08/2026\n\nMÁQUINA A6 VEL: 170\nDRT: 9363\...,SB-1579-N1N


# Análises

In [5]:
# ============================================================
# CONTADOR DE QUANTIDADE DE NOTAS POR HORA
# ============================================================

# Converter Data para datetime
df["Data"] = pd.to_datetime(
    df["Data"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

# Criar variável representando a hora do registro
df["Data_Hora"] = df["Data"].dt.floor("h")


# ============================================================
# AGRUPAR QUANTIDADE DE NOTAS POR HORA
# ============================================================

df_notas_hora = (
    df
    .groupby("Data_Hora")
    .size()
    .reset_index(name="Qntd_Notas_Hora")
    .sort_values("Data_Hora")
)


# ============================================================
# GRÁFICO
# ============================================================

fig = px.line(
    df_notas_hora,
    x="Data_Hora",
    y="Qntd_Notas_Hora",
    title="Quantidade de Notas por Hora",
    markers=True
)

fig.update_layout(
    xaxis_title="Data / Hora",
    yaxis_title="Quantidade de Notas"
)

fig.update_xaxes(
    tickformat="%H:%M\n%d/%m/%Y"
)

fig.show()

In [ ]:



# ============================================================
# 1. PREPARAÇÃO DOS DADOS
# ============================================================

# Converter Data para datetime
df["Data"] = pd.to_datetime(
    df["Data"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

# Criar variável representando a hora do registro
df["Data_Hora"] = df["Data"].dt.floor("h")


# ============================================================
# 2. CONTAGEM DE NOTAS
#    HORA + MÁQUINA + PREFIXO
# ============================================================

df_notas_hora = (
    df
    .groupby(
        ["Data_Hora", "Maquina", "Prefixo"],
        as_index=False
    )
    .size()
    .rename(
        columns={"size": "Qntd_Notas_Hora"}
    )
    .sort_values(
        ["Maquina", "Prefixo", "Data_Hora"]
    )
)


# ============================================================
# 3. DROPDOWN DE MÁQUINAS
# ============================================================

maquinas = sorted(
    df_notas_hora["Maquina"]
    .dropna()
    .unique()
)

dropdown_maquina = Dropdown(
    options=maquinas,
    value=maquinas[0],
    description="Máquina:"
)


# ============================================================
# 4. FUNÇÃO PARA BUSCAR PREFIXOS DA MÁQUINA
# ============================================================

def obter_prefixos(maquina):
    
    prefixos = (
        df_notas_hora.loc[
            df_notas_hora["Maquina"] == maquina,
            "Prefixo"
        ]
        .dropna()
        .unique()
    )

    return sorted(prefixos)


# Prefixos da primeira máquina
prefixos_iniciais = obter_prefixos(
    dropdown_maquina.value
)


# ============================================================
# 5. DROPDOWN DE PREFIXOS
# ============================================================

dropdown_prefixo = Dropdown(
    options=["Todos"] + prefixos_iniciais,
    value="Todos",
    description="Prefixo:"
)


# ============================================================
# 6. ÁREA DE SAÍDA DO GRÁFICO
# ============================================================

output_grafico = Output()


# ============================================================
# 7. FUNÇÃO PARA ATUALIZAR O GRÁFICO
# ============================================================

def atualizar_grafico():

    maquina = dropdown_maquina.value
    prefixo = dropdown_prefixo.value

    # Filtrar máquina
    df_grafico = df_notas_hora[
        df_notas_hora["Maquina"] == maquina
    ].copy()


    # ========================================================
    # CASO 1: TODOS OS PREFIXOS
    # ========================================================

    if prefixo == "Todos":

        fig = px.line(
            df_grafico,
            x="Data_Hora",
            y="Qntd_Notas_Hora",
            color="Prefixo",
            markers=True,
            title=f"Quantidade de Notas por Hora - Máquina {maquina}",
            hover_data=[
                "Maquina",
                "Prefixo"
            ]
        )


    # ========================================================
    # CASO 2: PREFIXO ESPECÍFICO
    # ========================================================

    else:

        df_grafico = df_grafico[
            df_grafico["Prefixo"] == prefixo
        ].copy()

        fig = px.line(
            df_grafico,
            x="Data_Hora",
            y="Qntd_Notas_Hora",
            markers=True,
            title=(
                f"Quantidade de Notas por Hora"
                f" - Máquina {maquina}"
                f" - Prefixo {prefixo}"
            ),
            hover_data=[
                "Maquina",
                "Prefixo"
            ]
        )


    # ========================================================
    # FORMATAÇÃO
    # ========================================================

    fig.update_layout(
        xaxis_title="Data / Hora",
        yaxis_title="Quantidade de Notas",
        hovermode="x unified"
    )

    fig.update_xaxes(
        tickformat="%H:%M\n%d/%m/%Y"
    )

    # Limpar gráfico anterior e mostrar novo
    with output_grafico:
        output_grafico.clear_output(wait=True)
        fig.show()


# ============================================================
# 8. ATUALIZAR PREFIXOS QUANDO MÁQUINA MUDAR
# ============================================================

def maquina_alterada(change):

    if change["name"] == "value":

        maquina = change["new"]

        # Buscar somente prefixos existentes para essa máquina
        novos_prefixos = obter_prefixos(maquina)

        # Atualizar opções
        dropdown_prefixo.options = [
            "Todos"
        ] + novos_prefixos

        # Voltar para "Todos"
        dropdown_prefixo.value = "Todos"

        # Atualizar gráfico
        atualizar_grafico()


# ============================================================
# 9. ATUALIZAR GRÁFICO QUANDO PREFIXO MUDAR
# ============================================================

def prefixo_alterado(change):

    if change["name"] == "value":
        atualizar_grafico()


# ============================================================
# 10. CONECTAR EVENTOS
# ============================================================

dropdown_maquina.observe(
    maquina_alterada,
    names="value"
)

dropdown_prefixo.observe(
    prefixo_alterado,
    names="value"
)


# ============================================================
# 11. EXIBIR FILTROS E GRÁFICO
# ============================================================

display(
    VBox([
        HBox([
            dropdown_maquina,
            dropdown_prefixo
        ]),
        output_grafico
    ])
)


# Mostrar gráfico inicial
atualizar_grafico()

In [7]:
df_maq_a2 = df.loc[df['Maquina'] == 'A2']

display(df_maq_a2.loc[df_maq_a2['Prefixo'] == 'SB-1036-SWN'])
display(df_maq_a2.loc[df_maq_a2['Prefixo'] == 'SB-1037-SWN'])

,Turno Wht,Maquina,Data,Funcao,Autor,Nota,Prefixo,Data_Hora
16,Manhã,A2,2026-08-10 08:19:58,Operador AQ,AQA2 Entoni,"inicio de troca, todas as ferramentas ok, fala...",SB-1036-SWN,2026-08-10 08:00:00
17,Manhã,A2,2026-08-10 12:49:07,Lider de turno,WILSON NOBRE,OBS. MAQ. PARADA PARA TM . AS 08;00 HS FORAM T...,SB-1036-SWN,2026-08-10 12:00:00
18,Manhã,A2,2026-08-10 13:27:11,Lider de turno,WILSON NOBRE,FOI INSTALADO O REDUTOR DE PESO NO TUBO,SB-1036-SWN,2026-08-10 13:00:00
19,Manhã,A2,2026-08-10 13:51:17,Esc. Automatica,ADAILSON,"CANAIS INSTALADOS MÁQUINA M;\nP. 1.5: RTE, RAB...",SB-1036-SWN,2026-08-10 13:00:00
125,Tarde,A2,2026-08-10 19:07:22,Operador AQ,AQA2 Wagner Leite,"Obs: N°12 com A+ foi corrigido, após amostra d...",SB-1036-SWN,2026-08-10 19:00:00
126,Tarde,A2,2026-08-10 19:17:49,Lider de turno,EDUARDO,FO FEITO UMA ONSPEÇÃO NA MÁQ M C/ 20 FRASCOS J...,SB-1036-SWN,2026-08-10 19:00:00
127,Tarde,A2,2026-08-10 19:21:06,Operador AQ,AQA2 Wagner Leite,Obs: Desviando a produção para o latão das 19:...,SB-1036-SWN,2026-08-10 19:00:00
128,Tarde,A2,2026-08-10 20:33:20,Lider AF,MIRANDA,"FIZ UM ACOMPAMHAMENTO NA MAQ M, DE 20 FRASCOS ...",SB-1036-SWN,2026-08-10 20:00:00
129,Tarde,A2,2026-08-10 21:16:30,Lider de turno,WILSON NOBRE,RELATÓRIO DE TROCA\n\n10/08/26 (A) | A2 | SB -...,SB-1036-SWN,2026-08-10 21:00:00
130,Tarde,A2,2026-08-10 21:16:42,Lider de turno,WILSON NOBRE,>>> Obs (T2) >>>\nS/ obs no (T2)\n\n>>> Obs ge...,SB-1036-SWN,2026-08-10 21:00:00


,Turno Wht,Maquina,Data,Funcao,Autor,Nota,Prefixo,Data_Hora
10,Manhã,A2,2026-08-10 06:04:08,Lider AF,PAMELY A2,DURANTE TODO O TURNO TIVEMOS MUITO PROBLEMA C...,SB-1037-SWN,2026-08-10 06:00:00
11,Manhã,A2,2026-08-10 06:25:27,Esc. Automatica,VINICIUS,"CANAIS INSTALADOS MÁQUINA M;\nP. 1.5: RTE, RAB...",SB-1037-SWN,2026-08-10 06:00:00
12,Manhã,A2,2026-08-10 07:01:58,Operador AQ,AQA2 Entoni,06h40 - Desviando cavidade n°12 com TP+. Foi f...,SB-1037-SWN,2026-08-10 07:00:00
13,Manhã,A2,2026-08-10 07:10:43,Operador AQ,AQA2 Entoni,06h40 as 07h05 - Foi feito o desvio do n°12 co...,SB-1037-SWN,2026-08-10 07:00:00
14,Manhã,A2,2026-08-10 07:34:10,Lider AF,SANTOS,7h30 MÁQ M descartando 4.3% amostra de 10 fras...,SB-1037-SWN,2026-08-10 07:00:00
15,Manhã,A2,2026-08-10 08:19:38,Lider AF,SANTOS,8h emp. real 75% ( erro de apontamento no tabl...,SB-1037-SWN,2026-08-10 08:00:00
